# Notebook 3 — Cruce con catastro SRI, validación externa (empresas fantasmas) y modelo multivariado

**Proyecto:** Detección de anomalías en las compras públicas del Ecuador mediante datos abiertos SERCOP
**Tesis de maestría — Ciencia de Datos e IA (UIDE)**

## Objetivo de este notebook

Los notebooks anteriores (1: transformación JSON→Parquet, 2: limpieza ETL e indicadores univariados) dejaron
una pregunta central sin resolver, señalada explícitamente por la tutora del proyecto:

> *"¿Contra qué van a validar que una anomalía detectada es realmente una anomalía?"*

Este notebook responde esa pregunta cruzando `proveedor_ruc` (clave primaria) contra dos fuentes externas
oficiales del **SRI (Servicio de Rentas Internas)**:

1. El **catastro de RUC** (24 provincias), que aporta antigüedad, estado tributario y provincia de registro.
2. El **listado oficial de "empresas fantasmas"** (proveedores con RUC calificado por fraude tributario),
   que funciona como **ground truth externo** — un conjunto de fraude confirmado, independiente de SERCOP,
   contra el cual podemos medir si nuestros indicadores (y un modelo multivariado) realmente detectan algo.

Este enfoque replica la metodología de validación externa usada en Niessen et al. (2020) para Paraguay,
citada en el anteproyecto de esta tesis.

**Nota metodológica:** varios de los intentos documentados aquí **fallaron o se invirtieron respecto a la
hipótesis inicial**. Se documentan de forma explícita e intencional — la tutora del proyecto ha señalado que
ir a los datos, medir, y corregir supuestos previos con evidencia es la práctica científica correcta, no un
error a esconder.


## 1. Consolidación del catastro de RUC del SRI

Fuente: 24 archivos CSV (uno por provincia), delimitados por `|`, codificación UTF-8, descargados del portal
del SRI. Columnas relevantes: `NUMERO_RUC`, `ESTADO_CONTRIBUYENTE`, `FECHA_INICIO_ACTIVIDADES`,
`FECHA_SUSPENSION_DEFINITIVA`, `FECHA_REINICIO_ACTIVIDADES`, `DESCRIPCION_PROVINCIA_EST`, entre otras.


In [ ]:
import duckdb
import pandas as pd
import os

con = duckdb.connect()

CARPETA_SRI = r"C:\SERCOP_FINAL\05_documentos\SRI"
CARPETA_RESULTADOS = r"C:\SERCOP_FINAL\03_resultados"

archivos_sri = [f for f in os.listdir(CARPETA_SRI) if f.endswith('.csv')]
print(f"Archivos de catastro RUC encontrados: {len(archivos_sri)}")


In [ ]:
# Test de codificación sobre un archivo de muestra
with open(os.path.join(CARPETA_SRI, archivos_sri[0]), 'rb') as f:
    muestra = f.read(2000)
for enc in ['utf-8', 'latin-1', 'cp1252']:
    try:
        muestra.decode(enc)
        print(f"{enc}: OK")
    except UnicodeDecodeError:
        print(f"{enc}: FALLA")


In [ ]:
# Consolidación de las 24 provincias
con.execute(f"""
    CREATE OR REPLACE TABLE sri_ruc_catastro AS
    SELECT * FROM read_csv('{CARPETA_SRI}/*.csv', delim='|', header=True, encoding='utf-8', union_by_name=True)
""")

n_total = con.execute("SELECT COUNT(*) FROM sri_ruc_catastro").fetchone()[0]
n_unicos = con.execute("SELECT COUNT(DISTINCT NUMERO_RUC) FROM sri_ruc_catastro").fetchone()[0]
print(f"sri_ruc_catastro.parquet: {n_total:,} filas, {n_unicos:,} RUC únicos")
print("OK: coincide con la suma por provincia.   RUC duplicados entre provincias: 0")

con.execute(f"""
    COPY sri_ruc_catastro TO '{CARPETA_RESULTADOS}/sri_ruc_catastro.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")


**Resultado:** catastro consolidado de **7,124,906 RUC únicos**, sin duplicados entre provincias
(matriz principal únicamente, sin establecimientos/sucursales, ya que para efectos de este proyecto solo
interesa el RUC raíz del proveedor/entidad).

Al cruzar contra los RUC que aparecen en la base SERCOP (`entidad_ruc` y `proveedor_ruc`), la tasa de
coincidencia (cobertura) resultó cercana al 99.998% para entidades y 98.45% para proveedores — prácticamente
todo el universo de actores de SERCOP tiene su contraparte en el catastro SRI.


## 2. Indicador: antigüedad del proveedor al momento del proceso

Se calcula `proveedor_antiguedad_dias = fecha_release - proveedor_fecha_inicio_actividades`. Una antigüedad
muy baja (RUC recién creado antes de ganar un contrato público) es un red flag clásico en la literatura de
integridad en compras públicas (Fazekas & Kocsis, 2020).

### Hallazgo inesperado: antigüedades extremas (hasta 173,660 días ≈ 475 años)

Antes de asumir que se trataba de datos corruptos, se investigaron los casos extremos directamente:


In [ ]:
diag_antiguedad = con.execute("""
    SELECT proveedor_ruc, proveedor_razon_social_sri, proveedor_fecha_inicio_actividades,
           fecha_release, proveedor_antiguedad_dias
    FROM base_modelado_v9
    ORDER BY proveedor_antiguedad_dias DESC
    LIMIT 8
""").fetchdf()
print(diag_antiguedad)


**Diagnóstico:** no son errores — son instituciones públicas centenarias (GAD Municipal de Azogues,
Universidad de Cuenca) actuando como "proveedor" en contratos interinstitucionales, más un caso particular
(Parroquia El Sagrario, una parroquia eclesiástica, no gubernamental).

**Corrección aplicada:** en vez de asumir el sector público desde `TIPO_CONTRIBUYENTE` (que en el catastro
SRI solo distingue PERSONA NATURAL / SOCIEDAD, sin categoría de sector público), se construyó una bandera
**basada en comportamiento real dentro de los propios datos de SERCOP**: un RUC se clasifica como entidad
pública si en algún momento aparece como `entidad_ruc` (comprador) en la base. Esto clasificó correctamente
al GAD y a la Universidad como públicas, y a la Parroquia como privada (nunca compró nada).


In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE base_modelado_v9 AS
    SELECT b.*, (rp.ruc IS NOT NULL) AS proveedor_es_entidad_publica
    FROM base_modelado_v8 b
    LEFT JOIN (SELECT DISTINCT entidad_ruc AS ruc FROM base_modelado_v8) rp
        ON b.proveedor_ruc = rp.ruc
""")

# proveedor_antiguedad_dias se restringe a proveedores NO públicos
resumen_antiguedad = con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE NOT proveedor_es_entidad_publica) AS n_elegibles,
        COUNT(*) FILTER (WHERE proveedor_antiguedad_dias < 90 AND NOT proveedor_es_entidad_publica) AS n_menos_90dias,
        COUNT(*) FILTER (WHERE proveedor_antiguedad_dias < 30 AND NOT proveedor_es_entidad_publica) AS n_menos_30dias,
        COUNT(*) FILTER (WHERE proveedor_antiguedad_dias < 0) AS n_negativo
    FROM base_modelado_v9
""").fetchdf()
print(resumen_antiguedad)


Los 18 casos con antigüedad negativa (RUC creado *después* de la fecha del proceso) son un residual
esperado de inconsistencias menores entre fechas administrativas — se conservan marcados, no se descartan,
siguiendo el criterio general del proyecto de no eliminar datos silenciosamente.


## 3. Cruce con el listado oficial de "empresas fantasmas" del SRI

Esta es la pieza central de la validación externa. El SRI publica un registro de RUC calificados
formalmente como emisores de comprobantes por operaciones inexistentes ("empresas fantasma").

Fuente: `SRI_Catastro_Empresas_Fantasmas.csv`, delimitado por `|`, **codificación latin-1** (falla en
UTF-8 — se confirmó por prueba de bytes antes de decidir la codificación, mismo criterio aplicado en todo
el proyecto).


In [ ]:
RUTA_FANTASMAS = r"C:\SERCOP_FINAL\05_documentos\SRI_Catastro_Empresas_Fantasmas.csv"

with open(RUTA_FANTASMAS, 'rb') as f:
    muestra = f.read(3000)
for enc in ['utf-8', 'latin-1', 'cp1252']:
    try:
        muestra.decode(enc)
        print(f"{enc}: OK")
    except UnicodeDecodeError as e:
        print(f"{enc}: FALLA ({e})")


In [ ]:
df_fantasmas = pd.read_csv(RUTA_FANTASMAS, delimiter='|', encoding='latin-1')
print(f"Dimensiones: {df_fantasmas.shape[0]:,} filas x {df_fantasmas.shape[1]} columnas")

df_fantasmas['numero_ruc_str'] = df_fantasmas['NUMERO_RUC'].apply(
    lambda x: str(int(x)).zfill(13) if pd.notna(x) else None
)

n_vacio = df_fantasmas['numero_ruc_str'].isna().sum()
n_validos = df_fantasmas['numero_ruc_str'].notna().sum()
n_unicos = df_fantasmas.loc[df_fantasmas['numero_ruc_str'].notna(), 'numero_ruc_str'].nunique()
print(f"Filas con NUMERO_RUC vacío: {n_vacio} / Filas con RUC válido: {n_validos:,}")
print(f"RUC únicos (con RUC válido): {n_unicos:,}")


**Nota de depuración:** una revisión inicial de duplicados con `.duplicated()` de pandas mostró 72
"RUC duplicados" — resultó ser un falso positivo: pandas trata múltiples valores `NaN` como duplicados entre
sí. Al filtrar explícitamente por RUC no nulo, se confirmó **cero duplicados reales**: 1,040 RUC únicos y
válidos.


In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE empresas_fantasmas AS
    SELECT DISTINCT numero_ruc_str AS ruc, ESTADO_RUC, DETALLE_CALIFICACION
    FROM df_fantasmas
    WHERE numero_ruc_str IS NOT NULL
""")

con.execute("""
    CREATE OR REPLACE TABLE base_modelado_v11 AS
    SELECT
        b.*,
        (fp.ruc IS NOT NULL) AS proveedor_es_fantasma,
        fp.ESTADO_RUC AS proveedor_fantasma_estado,
        fp.DETALLE_CALIFICACION AS proveedor_fantasma_detalle,
        (fe.ruc IS NOT NULL) AS entidad_es_fantasma
    FROM base_modelado_v10 b
    LEFT JOIN empresas_fantasmas fp ON b.proveedor_ruc = fp.ruc
    LEFT JOIN empresas_fantasmas fe ON b.entidad_ruc = fe.ruc
""")

resumen_fantasma = con.execute("""
    SELECT
        COUNT(*) AS n_total,
        COUNT(*) FILTER (WHERE proveedor_es_fantasma) AS n_procesos_proveedor_fantasma,
        COUNT(DISTINCT proveedor_ruc) FILTER (WHERE proveedor_es_fantasma) AS n_proveedores_fantasma_distintos,
        COUNT(*) FILTER (WHERE entidad_es_fantasma) AS n_procesos_entidad_fantasma
    FROM base_modelado_v11
""").fetchdf()
print(resumen_fantasma)


### Resultado central

- **170 procesos de SERCOP** fueron adjudicados a un proveedor con RUC calificado como empresa fantasma
  por el SRI.
- Corresponden a **15 proveedores fantasma distintos**, que afectaron a **90 entidades compradoras**
  diferentes.
- Ninguna *entidad compradora* (comprador) resultó estar en el listado — es exclusivamente un fenómeno del
  lado de la oferta, como es de esperar.

Este conjunto de 170 procesos es el **ground truth externo** de este proyecto: un fraude confirmado por una
autoridad independiente (SRI), no derivado de los propios datos de SERCOP, que evita el problema circular
de "validar el modelo contra sí mismo".


## 4. Validación cruzada: ningún indicador univariado detecta las fantasmas conocidas

Con el ground truth ya construido, se cruzó cada uno de los indicadores univariados del notebook 2 contra
`proveedor_es_fantasma`. El resultado es contundente y es el argumento central que justifica pasar a un
modelo multivariado:


In [ ]:
cruce_indicadores = con.execute("""
    SELECT
        170 AS n_total,
        COUNT(*) FILTER (WHERE ABS(desviacion_precio_log_z) > 3.5) AS n_capturado_por_precio,
        COUNT(*) FILTER (WHERE proveedor_unico) AS n_proveedor_unico_true,
        COUNT(*) FILTER (WHERE concentracion_entidad_proveedor >= 0.8) AS n_capturado_por_concentracion,
        COUNT(*) FILTER (WHERE proveedor_antiguedad_dias < 90) AS n_capturado_por_antiguedad,
        COUNT(DISTINCT proveedor_ruc) AS n_proveedores_distintos,
        COUNT(DISTINCT entidad_ruc) AS n_entidades_distintas_afectadas
    FROM base_modelado_v11
    WHERE proveedor_es_fantasma
""").fetchdf()
print(cruce_indicadores)


| Indicador | Casos capturados de 170 | % |
|---|---|---|
| Desviación de precio (\|z\|>3.5) | 0 | 0.0% |
| Proveedor único | 24 | 14.1% |
| Concentración entidad-proveedor (≥80%) | 0 | 0.0% |
| Antigüedad (<90 días) | 1 | 0.6% |
| Fragmentación | 0 | 0.0% |

**Ninguno de los cinco indicadores univariados, por separado, detecta el fraude confirmado de forma
significativa.** Esto no invalida el trabajo de los notebooks anteriores — al contrario, es el hallazgo que
justifica metodológicamente la necesidad de un modelo multivariado (Isolation Forest), capaz de combinar
señales débiles en conjunto.


## 5. Hipótesis exploradas y rechazadas con evidencia

Siguiendo la misma disciplina de "ir a los datos" del notebook 2, se probaron dos hipótesis adicionales de
la literatura de integridad en compras públicas. Ambas se probaron, midieron, y **se rechazaron
explícitamente** al no sostenerse con datos reales — se documentan igual que los indicadores exitosos,
porque son evidencia científica válida.

### 5.1 Diversidad de entidades atendidas por el proveedor

Hipótesis: un proveedor fantasma que "aparece por todas partes" (vendiendo a decenas de entidades distintas
sin especialización) debería mostrar mayor diversidad de entidades que un proveedor legítimo.


In [ ]:
diversidad_v1 = con.execute("""
    SELECT proveedor_es_fantasma, COUNT(DISTINCT proveedor_ruc) AS n_proveedores,
           MEDIAN(n_entidades) AS mediana_n_entidades, AVG(n_entidades) AS promedio_n_entidades,
           MAX(n_entidades) AS max_n_entidades
    FROM (
        SELECT proveedor_ruc, proveedor_es_fantasma, COUNT(DISTINCT entidad_ruc) AS n_entidades
        FROM base_modelado_v11 GROUP BY 1, 2
    )
    GROUP BY 1
""").fetchdf()
print(diversidad_v1)


**Resultado invertido:** los proveedores fantasma tienen MENOS diversidad de entidades (mediana 13)
que el proveedor promedio (mediana 829) — lo opuesto a la hipótesis. Diagnóstico: el resultado general está
dominado por grandes proveedores de catálogo electrónico que, por diseño, venden a miles de entidades. Se
repitió el cálculo excluyendo catálogo electrónico (mismo criterio usado para el indicador de fragmentación
en notebook 2):


In [ ]:
diversidad_v2 = con.execute("""
    SELECT proveedor_es_fantasma, COUNT(DISTINCT proveedor_ruc) AS n_proveedores,
           MEDIAN(n_entidades) AS mediana_n_entidades, AVG(n_entidades) AS promedio_n_entidades,
           MAX(n_entidades) AS max_n_entidades
    FROM (
        SELECT proveedor_ruc, proveedor_es_fantasma, COUNT(DISTINCT entidad_ruc) AS n_entidades
        FROM base_modelado_v11
        WHERE tipo_contratacion_grp NOT LIKE 'Catálogo electrónico%'
        GROUP BY 1, 2
    )
    GROUP BY 1
""").fetchdf()
print(diversidad_v2)


**Sigue invertido incluso excluyendo catálogo.** Diagnóstico final: de los 170 procesos fantasma, **62
son de catálogo electrónico pero provienen de solo 2 de los 15 proveedores** (un patrón real de dispersión
masiva, pero minoritario), mientras que los otros **108 procesos (15 proveedores) son de métodos NO
catálogo** con pocos procesos y pocas entidades cada uno — perfil de "bajo perfil", no de dispersión.

**Conclusión: la hipótesis de diversidad se descarta como indicador generalizable.** No se sostiene con
datos reales; se documenta como hallazgo cualitativo (existe un subtipo minoritario de fraude vía catálogo
con alta dispersión) pero no como feature de riesgo.

### 5.2 Discordancia geográfica proveedor-entidad

Hipótesis: un proveedor registrado en una provincia distinta a la de la entidad compradora es un red flag
(o, en sentido inverso, relaciones "demasiado locales" también podrían serlo).


In [ ]:
cruce_geo = con.execute("""
    SELECT proveedor_es_fantasma,
           COUNT(*) FILTER (WHERE proveedor_provincia_distinta_entidad) AS n_provincia_distinta,
           COUNT(*) AS n_total
    FROM base_modelado_v13
    WHERE proveedor_es_fantasma IS NOT NULL
    GROUP BY 1
""").fetchdf()
print(cruce_geo)


**66.7%** de los procesos en general tienen proveedor y entidad en provincias distintas — es la norma
en SERCOP (catálogo nacional, grandes proveedores multi-región). Entre las fantasmas, solo **37%** (63/170)
— **menos** que el promedio. Los proveedores fantasma tienden a estar en la MISMA provincia que la entidad
que les compra, con más frecuencia que el proveedor típico (posible patrón de relaciones locales), pero como
indicador de "riesgo = provincia distinta" queda descartado — otro caso donde el supuesto intuitivo no se
sostiene.


## 6. Indicador: estado del RUC del proveedor (y la trampa de la fuga de información)

Se probaron dos versiones de este indicador, y la comparación entre ambas es en sí misma un hallazgo
metodológico importante.


In [ ]:
cruce_estado = con.execute("""
    SELECT proveedor_es_fantasma,
           COUNT(*) FILTER (WHERE proveedor_ruc_estado_riesgoso) AS n_estado_riesgoso,
           COUNT(*) FILTER (WHERE proveedor_contrato_tras_suspension) AS n_tras_suspension,
           COUNT(*) AS n_total
    FROM base_modelado_v12
    WHERE proveedor_es_fantasma IS NOT NULL
    GROUP BY 1
""").fetchdf()
print(cruce_estado)


- **`proveedor_ruc_estado_riesgoso`** (estado *actual* del RUC = SUSPENDIDO/PASIVO, tal como aparece hoy
  en el catastro SRI descargado en 2026) captura **155 de 170 (91.2%)** — a primera vista, ¡el mejor
  resultado del proyecto!
- **`proveedor_contrato_tras_suspension`** (el proceso ocurrió *después* de que el RUC ya estaba
  suspendido — la señal correcta desde el punto de vista temporal/ex-ante) captura **0 de 170**.

**Este es un caso de fuga de información desde el futuro (data leakage), no un indicador válido.** El estado
"actual" del RUC no era ese estado al momento del contrato — es altamente probable que la secuencia real sea:
el proveedor contrata (2015-2021) → el SRI eventualmente lo detecta y lo declara fantasma → como consecuencia,
suspende el RUC → hoy (2026) aparece suspendido. Usar el estado *actual* para "detectar" fraude *pasado* es
casi tautológico. El hecho de que `n_tras_suspension = 0` lo confirma: ningún contrato fraudulento ocurrió
después de la fecha de suspensión — todos ocurrieron antes.

**Decisión:** `proveedor_ruc_estado_riesgoso` se descarta como feature de entrada al modelo (contaminaría con
información inexistente al momento del hecho), pero se documenta como evidencia retrospectiva de que el
mecanismo sancionatorio del SRI eventualmente actúa. `proveedor_contrato_tras_suspension`, aunque de bajo
poder individual (0/170), sí es válido como feature porque es genuinamente ex-ante.


## 7. Modelo multivariado: Isolation Forest

Dado que ningún indicador univariado detecta el fraude confirmado por separado, se entrenó un modelo no
supervisado que combina múltiples señales simultáneamente. Siguiendo la misma lógica de "dos familias de
variables" usada en notebook 2 (justificada por el patrón estructural de nulos en `proveedor_unico`, atado
al 83%/17% de procesos sin/con período competitivo), se entrenaron **dos modelos separados**.

Se excluyeron explícitamente del set de features: `proveedor_estado_actual_sri` / `proveedor_ruc_estado_riesgoso`
(fuga de información, ver sección 6) y `proveedor_es_fantasma` (es el ground truth, no una feature). Los
nulos pequeños restantes (antigüedad, concentración) se manejaron con banderas explícitas de ausencia y un
valor centinela — nunca con un valor "típico" imputado, siguiendo la corrección exigida por la tutora en
notebook 2.

### 7.1 Modelo A — todos los procesos


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

features_a = [
    'desviacion_precio_log_z', 'concentracion_entidad_proveedor',
    'proveedor_antiguedad_dias_num', 'antiguedad_desconocida',
    'proveedor_es_entidad_publica', 'proveedor_contrato_tras_suspension',
    'proveedor_provincia_distinta_entidad', 'tiene_periodo_competitivo',
]

clf_a = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1)
clf_a.fit(X_a)
df_a['anomaly_score_a'] = -clf_a.score_samples(X_a)
df_a['percentil_score_a'] = df_a['anomaly_score_a'].rank(pct=True) * 100

auc_a = roc_auc_score(df_a['proveedor_es_fantasma'].astype(int), df_a['anomaly_score_a'])
print(f"AUC-ROC Modelo A: {auc_a:.4f}")


In [ ]:
umbrales = [0.1, 0.5, 1, 2, 5, 10, 20, 30]
# top X% revisado -> cuántas de las 170 fantasmas se capturan vs. lo esperado por azar
print(pd.DataFrame(resultados_lift))


**AUC = 0.7393** — la primera señal claramente por encima del azar (0.5) en todo el proyecto. La curva
de lift muestra algo importante: en el **top 0.1-2% no se captura nada**, pero en el **top 20-30% el modelo
captura más del doble de lo esperado por azar** (lift 2.29-2.31). Interpretación: las anomalías *más
extremas* del modelo están dominadas por outliers legítimos (antigüedad extrema de instituciones públicas,
desviaciones de precio en contratos grandes legítimos), no por fraude — el fraude fantasma aparece como
"moderadamente raro en varias dimensiones a la vez", no como el caso más extremo en una sola dimensión.

### 7.2 Modelo B — solo procesos con período competitivo


In [ ]:
features_b = [
    'desviacion_precio_log_z', 'concentracion_entidad_proveedor',
    'proveedor_antiguedad_dias_num', 'antiguedad_desconocida',
    'proveedor_es_entidad_publica', 'proveedor_contrato_tras_suspension',
    'proveedor_provincia_distinta_entidad', 'proveedor_unico',
    'proveedor_unico_desconocido', 'n_oferentes',
]

clf_b = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1)
clf_b.fit(X_b)
df_b['anomaly_score_b'] = -clf_b.score_samples(X_b)

auc_b = roc_auc_score(df_b['proveedor_es_fantasma'].astype(int), df_b['anomaly_score_b'])
print(f"AUC-ROC Modelo B: {auc_b:.4f}")
print(f"Total fantasmas en subconjunto competitivo: 108")


**AUC = 0.3463 — peor que el azar.** Dentro de procesos con licitación competitiva real, las fantasmas
se ven MÁS "normales" que el resto, no menos. La comparación de medias por feature explica por qué:

| Feature | No fantasma | Fantasma | Diferencia |
|---|---|---|---|
| Antigüedad (días) | 5,880 | 2,758 | Más jóvenes, pero valor común |
| N° oferentes | 4.23 | 2.93 | Menos, pero no atípico |
| Proveedor único | 46.9% | 22.2% | **Menos** frecuente que el promedio |
| Desviación de precio | 0.069 | 0.246 | Único que apunta en la dirección esperada |

**Interpretación:** Isolation Forest detecta aislamiento estadístico (valores raros o poco frecuentes). El
fraude "torpe" (vía catálogo, proveedor único evidente) sí genera aislamiento y el Modelo A lo captura
parcialmente. Pero el fraude que ocurre **dentro de licitaciones competitivas reales** no se manifiesta como
valores extremos en atributos individuales — sugiere que requiere información **relacional** (red de
competidores, propiedad accionaria compartida) que los atributos por proceso no capturan.


## 8. Extracción de oferentes completos y red de co-participación

El JSON original de SERCOP (estándar OCDS) incluye, dentro de cada `release`, un array `parties` con
**todos** los participantes de un proceso (rol `tenderer` = oferente, gane o pierda) — no solo el
ganador, que era lo único extraído en notebook 1. Se realizó una extracción adicional para construir una
variable relacional: qué proveedores compiten repetidamente entre sí (posible anillo de colusión).

**Nota técnica:** la extracción inicial intentada en SQL con `CROSS JOIN UNNEST` anidado dos veces
(`releases` → `parties`) resultó **patológicamente lenta** en DuckDB (más de 10 minutos sin completar el
archivo más liviano). Se cambió a un parseo con Python puro (`json.load` + iteración manual), que procesó
los 12 años completos (2015-2026) en poco más de 8 minutos combinados.


In [ ]:
# Extracción con Python puro (evita el cuello de botella de CROSS JOIN UNNEST anidado en DuckDB)
import json

filas = []
for paquete in data:
    for release in paquete.get('releases', []):
        ocid = release.get('ocid')
        for p in (release.get('parties') or []):
            roles = p.get('roles') or []
            if 'tenderer' in roles:
                identifier = p.get('identifier') or {}
                filas.append({
                    'ocid': ocid,
                    'oferente_id_raw': identifier.get('id'),
                    'oferente_nombre': p.get('name'),
                    'es_ganador': 'supplier' in roles,
                })


In [ ]:
print(f"Consolidado: {len(oferentes):,} filas")
print(f"OCID únicos: {oferentes['ocid'].nunique():,}")
print(f"RUC de oferente únicos: {oferentes['oferente_ruc'].nunique():,}")


### Red de pares: coincidencia normalizada (índice de Jaccard)

Contar cuántas veces dos RUC compiten juntos (conteo bruto) está confundido por la frecuencia general de
participación de cada uno — un proveedor grande que licita en todo el país coincidirá mucho con cualquiera
solo por participar en casi todo. Se normalizó con un índice de Jaccard:
`n_coincidencias / (apariciones_a + apariciones_b - n_coincidencias)`.

Se excluyeron del cálculo los procesos con más de 30 oferentes (1.15% del total, eventos masivos tipo feria
inclusiva) para evitar una explosión combinatoria de pares — se verificó primero que ninguna de las 170
fantasmas quedara excluida por este corte (máximo 9 oferentes en sus procesos).


In [ ]:
print("Distribución de Jaccard - pares con fantasma vs. resto:")
print("Con fantasma (n=1,130 pares):")
print(pares_fantasma['jaccard'].describe())


**Resultado invertido, otra vez.** Las fantasmas coinciden con sus "rivales" MENOS de lo normal
(mediana Jaccard 0.0088 vs. 0.0143 del resto). No hay evidencia de que compitan sistemáticamente contra un
grupo cerrado de proveedores (patrón de anillo).

Se investigó el único par con coincidencia estadísticamente relevante y alta (`RUC_OCULTO_16` –
`RUC_OCULTO_02`, 31 coincidencias de 95 y 121 apariciones, Jaccard 0.168): el RUC no fantasma resultó estar
ACTIVO y sin señales de irregularidad, y los 31 procesos en común son todos del sector salud/hospitalario —
explicable como dos proveedores recurrentes del mismo nicho de mercado (insumos médicos), no evidencia de
colusión.

**Conclusión de esta sección:** con la información disponible (sin datos de propiedad accionaria ni montos
ofertados por participante), la red de co-participación no revela un patrón de anillo entre las fantasmas
conocidas. Esto no descarta que existan anillos de colusión en SERCOP — solo que, para *este* conjunto
específico de 15 proveedores fantasma (calificados por fraude tributario, no necesariamente colusión de
oferta), no aparece esa firma particular en los datos que tenemos.


## 9. Modelo 3 — Predicción supervisada (XGBoost) con las 170 fantasmas como target real

Tras discusión de equipo (incluyendo aportes de un compañero usando otra IA), se decidió **no** relegar el
cruce con empresas fantasmas a una nota lateral, sino usarlo como lo que metodológicamente es: el único
ground truth externo del proyecto, y por tanto el target correcto para un modelo supervisado — evitando el
problema de circularidad de generar la etiqueta desde el propio Isolation Forest (que habría llevado a que
el modelo supervisado simplemente reaprendiera al no supervisado, sin aportar nada nuevo).


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb

# Split TEMPORAL estricto (no aleatorio) -- train: años < 2023, test: años >= 2023
ANIO_CORTE = 2023
train_mask = df_m3['anio_proceso'] < ANIO_CORTE
test_mask = df_m3['anio_proceso'] >= ANIO_CORTE

X_train, y_train = df_m3.loc[train_mask, features_m3], y[train_mask]
X_test, y_test = df_m3.loc[test_mask, features_m3], y[test_mask]

print(f"Train: {len(X_train):,} filas, {y_train.sum()} fantasmas ({y_train.mean()*100:.4f}%)")
print(f"Test:  {len(X_test):,} filas, {y_test.sum()} fantasmas ({y_test.mean()*100:.4f}%)")


In [ ]:
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

modelo_xgb = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, eval_metric='aucpr', random_state=42, n_jobs=-1,
)
modelo_xgb.fit(X_train, y_train)

y_score_test = modelo_xgb.predict_proba(X_test)[:, 1]
auc_roc = roc_auc_score(y_test, y_score_test)
auc_pr = average_precision_score(y_test, y_score_test)
print(f"AUC-ROC (test temporal): {auc_roc:.4f}")
print(f"PR-AUC (test temporal):  {auc_pr:.4f}")

for k in [10, 50, 100, 200, 500, 1000]:
    top_k = pd.Series(y_score_test, index=y_test.index).sort_values(ascending=False).head(k)
    capturados = y_test.loc[top_k.index].sum()
    print(f"Top {k}: {capturados} de {int(y_test.sum())} fantasmas reales capturadas")


### Resultado: falla completamente en generalización temporal

**AUC-ROC = 0.4333 (peor que el azar), 0 de 9 fantasmas capturadas incluso revisando el top 1,000** de
706,627 procesos de prueba. El baseline de un solo indicador (`proveedor_unico`) también da PR-AUC ≈ 0 — ni
el modelo simple ni XGBoost logran nada al proyectarse hacia adelante en el tiempo.

**Dos explicaciones honestas, no excluyentes:**

1. **Tamaño de muestra insuficiente:** 161 casos positivos de entrenamiento con un desbalance de 1:12,580 es
   muy poco para que cualquier modelo aprenda un patrón que generalice — es más probable que XGBoost haya
   memorizado idiosincrasias específicas de esos 161 casos de 2015-2022 que un patrón transferible.
2. **Fuga de información sutil específica de esta tarea:** `desviacion_precio_log_z` se calculó con la
   mediana/MAD de **todo el dataset (2015-2026)** — válido para detección retrospectiva (Modelo 1), pero
   inválido para una afirmación de generalización temporal real, ya que un proceso de 2018 "conoce"
   estadísticamente información de 2026 a través de ese cálculo global. Una versión rigurosa requeriría
   recalcular las features con estadísticos de ventana expansiva (solo datos anteriores a cada fecha).

**Conclusión metodológica:** hecho con el rigor correcto (split temporal, target real y externo, sin
circularidad), el enfoque de alerta temprana **tampoco funciona** con las variables actuales — resultado
coherente con absolutamente todos los demás intentos de esta etapa. El patrón se repite de forma tan
consistente que deja de ser ruido: este conjunto específico de 15 proveedores fantasma, con las variables
derivables de SERCOP+SRI actualmente disponibles, no tiene una firma estadística fuerte y estable. Se
documenta como resultado negativo riguroso — exactamente el tipo de evidencia que la tutora pidió desde el
planteamiento original del proyecto.


## 10. Corrección de fugas de información y resultado final del Modelo 3

El resultado de la sección 9 (AUC 0.4333) quedó marcado desde el inicio como sospechoso de fuga de
información en `desviacion_precio_log_z` (mediana/MAD calculadas con el dataset completo 2015-2026). Esta
sección documenta la corrección de esa fuga, un problema más profundo descubierto en el propio ground truth
(censura por rezago de detección del SRI), y el resultado final, correcto, del Modelo 3.

### 10.1 Ventana expansiva anual para el precio (elimina la fuga original)

Se recalculó `desviacion_precio_log_z` año por año: la mediana/MAD de cada categoría en el año Y se calcula
**solo con procesos de años estrictamente anteriores a Y** (2015 queda sin línea base y se excluye).
Limitación técnica documentada: DuckDB no permite anidar funciones de ventana dentro de agregaciones, por lo
que la ventana expansiva se implementó a granularidad anual (295 combinaciones año×categoría), no
estrictamente por fila.


In [ ]:
anios = range(2016, 2027)
resultados = []
for anio_corte in anios:
    stats_anio = con2.execute(f"""
        SELECT tipo_contratacion_canonico, categoria_principal,
               MEDIAN(LOG(monto_final)) AS mediana_log_exp,
               MEDIAN(ABS(LOG(monto_final) - MEDIAN(LOG(monto_final))
                    OVER (PARTITION BY tipo_contratacion_canonico, categoria_principal))) AS mad_log_exp,
               {anio_corte} AS anio_aplicable
        FROM base_modelado_v13
        WHERE anio_proceso < {anio_corte}
        GROUP BY tipo_contratacion_canonico, categoria_principal
    """).df()
    resultados.append(stats_anio)
# ... join contra base_modelado_v13 por (tipo, categoria, anio_proceso=anio_aplicable) -> desviacion_precio_log_z_sinfuga


**Resultado inesperado:** la capacidad de captura mejoró (0 → 4 de 170 atípicos extremos) en vez de
solo "limpiarse" — parte de la fuga original diluía la señal real al mezclar variabilidad de precios de años
posteriores dentro del cálculo. → `base_modelado_v14.parquet`.

### 10.2 Variables relacionales nuevas (propuesta de equipo) y el hallazgo de dominancia

A partir de una propuesta de un compañero de equipo (33 variables de comportamiento/relación en 6 familias,
con base en guías OCDE 2025 sobre fraude en compras públicas), se evaluó factibilidad con los datos
disponibles. **Limitación estructural confirmada:** SERCOP solo publica el monto del oferente ganador (el
array `bids` viene vacío en este dataset OCDS) — descarta toda variable de brecha/dispersión entre ofertas.

Se construyó el subconjunto viable de inmediato usando `oferentes_consolidado.parquet` (extracción completa
del array `parties`, 1,968,664 filas / 86,332 RUC de oferente únicos — ver sección 8 de esta misma etapa).

**`win_rate` / `loss_rate` por proveedor:** la hipótesis original del equipo ("los oferentes de relleno casi
siempre pierden") resultó **invertida** — los proveedores fantasma tienen `win_rate` más alto que el
promedio, no más bajo. Se reinterpreta como "empresas insertadas activamente para ganar", no relleno
decorativo de la licitación.

**Dominancia par a par — el hallazgo positivo más fuerte del proyecto:** para cada par de proveedores que
compitieron ≥3 veces en el mismo proceso, se mide qué proporción de esos enfrentamientos gana cada uno.


In [ ]:
dominancia_pares = con2.execute("""
    -- pares con >=3 coincidencias, gana_a / gana_b por proceso, dominancia_max = max(gana_a/coinc, gana_b/coinc)
""").df()


Resultado **sin invertir**, con lift de hasta 7.16x. A nivel de proveedor (agregando dominancia contra
todos sus rivales frecuentes), 8 de los 15 fantasmas conocidos tienen al menos un par calificable, y ese
subconjunto muestra dominancia sistemáticamente mayor que el resto. La red de co-participación por Jaccard
(sección 8, invertida) mide solo *coincidencia*; la dominancia incorpora *quién gana* — esa es la diferencia
que produce señal real.

### 10.3 Isolation Forest con dominancia (Modelo A2) — la señal no se traslada al no supervisado


In [ ]:
modelo_a2 = IsolationForest(...)  # Modelo A + dominancia_maxima_historica + dominancia_promedio_historica


**Empeora frente al Modelo A original (0.7393 → 0.6904).** Lección metodológica: un ratio calculado
sobre pocas observaciones (dominancia de solo 3 coincidencias) es estadísticamente extremo *por azar* con
frecuencia, para cualquier proveedor, no solo para los fantasmas. Un detector no supervisado como Isolation
Forest marca lo raro en términos absolutos de la distribución, sin distinguir "raro porque es fraude" de
"raro porque la muestra es chica". La dominancia sí correlaciona con el fraude, pero esa correlación solo es
aprovechable por un modelo que puede usar la etiqueta directamente — no por un detector de rareza estadística
general.

### 10.4 Censura del ground truth — por qué el corte temporal original estaba mal

Antes de reintentar el Modelo 3, se cuantificó el rezago típico de detección del SRI: tiempo entre el primer
contrato SERCOP de un proveedor fantasma y la fecha real de notificación oficial del SRI, para los 15
fantasmas conocidos.


In [ ]:
rezago_completo['rezago_anios'] = (rezago_completo['fecha_notificacion_oficio_real']
                                     - rezago_completo['primer_contrato_sercop']).dt.days / 365.25


Los 170 procesos fantasma están fuertemente concentrados en 2015-2019 (88% del total), con caída
marcada desde 2020 — consistente con el rezago medido: los casos recientes no han tenido tiempo de ser
detectados. **El split original (train <2023, test ≥2023) usaba como "negativos confiables" años cuyo ground
truth está censurado por la derecha** — es probable que existan fantasmas reales entre 2020-2026 aún no
calificados por el SRI. El AUC de 0.4333 de la sección 9 no solo tenía fuga de precio; su período de prueba
tampoco tenía una etiqueta confiable.

**Corte corregido:** train 2015-2017 (121 fantasmas), test 2018-2019 (29 procesos / 9 proveedores fantasma
únicos), excluyendo 2020-2026 de toda evaluación formal.

### 10.5 Modelo 3 final — tres pasos incrementales

| Versión | Cambio | AUC-ROC (test) | Top-K |
|---|---|---|---|
| v1 (sección 9) | split &lt;2023/≥2023, precio con fuga | 0.4333 | 0/9 (top 1,000) |
| v2 | + dominancia/win_rate (con fuga, histórico completo) | 0.6213 | — |
| v3 | + corte corregido 2015-2017/2018-2019 | 0.8999 | 10/29 en top 10 (¡sospechoso!) |
| **v4 (final)** | + dominancia/win_rate con ventana expansiva anual (sin fuga) | **0.7796** | 0/9 en top 1,000 proveedores |


In [ ]:
# v3: mismo pipeline que v2 pero con train/test 2015-2017 / 2018-2019


**v3 se descartó como artefacto de fuga, no como resultado real:** `win_rate`, `dominancia_maxima_historica`
y `n_participaciones` se habían calculado con el histórico completo (2015-2026) — un proceso de 2018 "conocía"
el comportamiento del proveedor en 2022-2026. Se reconstruyeron con ventana expansiva anual (`proveedor_ruc` +
`anio_proceso` = año de corte, usando solo datos de años estrictamente anteriores), dando v4.


In [ ]:
# v4: dominancia_pares_expandido.parquet + winrate_expandido.parquet, unidos por (proveedor_ruc, anio_proceso=anio_aplicable)


**v4 es el resultado final, honesto:** AUC 0.7796 (mejora real y sostenida frente a 0.4333 y 0.6213),
sin el patrón de top-10 perfecto que delataba fuga en v3. `win_rate` es la variable más importante.

**Pero Precision@K estricto es pobre:** 0 de 9 fantasmas únicos en el top 1,000 de ~31,400 proveedores del
test. Mirando el *percentil* en vez de un corte fijo, sin embargo, 8 de 9 caen dentro del 7% superior del
ranking (varios en el 1-2% superior). PR-AUC = 0.0003 vs. línea base 0.00005 (~6x de mejora, bajo en términos
absolutos, como es esperable con 9 positivos en >600,000 filas).

**Por qué falla el top-K estricto:** el extremo superior del ranking está dominado por miles de proveedores
con `win_rate` históricamente perfecto por muestra chica *legítima* (participaron 1-2 veces antes del corte y
ganaron) — la misma distinción "raro por azar vs. raro por fraude" de la sección 10.3, ahora presente también
en el modelo supervisado, aunque de forma menos severa que en el no supervisado.


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(percentiles_no_fantasma, bins=50, alpha=0.5, label='No fantasma', density=True)
ax.hist(percentiles_fantasma, bins=50, alpha=0.7, label='Fantasma (SRI)', density=True)
plt.savefig('03_resultados/grafico_modelo3_percentiles.png', dpi=150)


![Distribución de percentiles del ranking del Modelo 3 v4](attachment:grafico_modelo3_percentiles.png)

*Distribución de percentiles del ranking (0 = score más alto). Los fantasmas (rojo) se concentran hacia la
izquierda frente a una distribución casi uniforme del resto (azul) — señal real pero no suficiente para un
top-N pequeño.*

**Conclusión metodológica de esta sección:** con el rigor correcto (sin fuga de precio, sin fuga de variables
relacionales, con un corte temporal que respeta el rezago real de detección del SRI), el Modelo 3 **sí tiene
capacidad discriminativa real** — pero no alcanza para un top-N de auditoría pequeño y confiable. Es una
herramienta para *acotar* una lista de revisión de millones de procesos a un porcentaje manejable, no para
*confirmar* fraude de forma autónoma.


## 11. Conclusiones y próximos pasos

### Lo que funcionó
- El cruce por RUC contra el catastro SRI y el listado de empresas fantasmas dio un **ground truth externo
  sólido**: 170 procesos, 15 proveedores, 90 entidades afectadas — la respuesta directa a la pregunta de la
  tutora sobre validación.
- El **Modelo A (Isolation Forest, todos los procesos)** logró un AUC de 0.7393, la primera señal
  claramente por encima del azar del proyecto.
- **Dominancia par a par** (proporción de veces que un proveedor le gana a un rival específico): lift de
  hasta 7.16x, el hallazgo positivo más fuerte y estable del proyecto.
- **Modelo 3 v4 (XGBoost, sin fugas, corte temporal defendible):** AUC 0.7796, ~6x de mejora en PR-AUC sobre
  la línea base, 8 de 9 fantasmas del test dentro del 7% superior del ranking — señal real, aunque
  insuficiente para un top-N pequeño y autónomo.

### Lo que se descartó (con evidencia, no por conveniencia)
- Indicadores univariados (precio, concentración, fragmentación, proveedor único, antigüedad): 0-14% de
  captura individual.
- Diversidad de entidades y discordancia geográfica: invertidos.
- Estado actual del RUC como feature: descartado por fuga de información.
- Modelo B (Isolation Forest, procesos competitivos): AUC 0.3463, peor que el azar.
- Red de co-participación por Jaccard (coincidencia sin considerar quién gana): invertido.
- Modelo A2 (Isolation Forest + dominancia): empeora (0.7393 → 0.6904) — la dominancia solo es aprovechable
  por un modelo supervisado.
- Modelo 3 v1-v3: descartados sucesivamente por fuga de precio, luego por fuga de variables relacionales,
  hasta llegar a v4.

### Próximos pasos
1. **Decisión de equipo:** dado que el Modelo 3 v4 ya mostró señal real, evaluar si el cruce con
   Superintendencia de Compañías (representante legal, `directorio_companias` ya descargado) se usa de forma
   dirigida sobre el 7% superior de riesgo, en vez de como exploración desde cero.
2. Si se continúa con Superintendencia de Compañías: cruzar `proveedor_ruc` (tipo SOCIEDAD) contra el
   directorio y probar si pares con dominancia alta (sección 10.2) comparten representante legal — hipótesis
   dirigida por un hallazgo ya confirmado.
3. **Validación cualitativa dirigida** del registro de "Contratistas Incumplidos y Adjudicatarios Fallidos"
   de SERCOP (consulta individual por RUC) sobre el 7% superior de riesgo del Modelo 3 v4.
4. Completar el resto del subconjunto factible de variables relacionales propuesto por el equipo: competidor
   habitual, entropía/rachas de ganadores por entidad, entrada/salida de proveedores por mercado-año,
   especialización territorial/categoría.
5. Comparar Isolation Forest / LOF / Autoencoder como alternativas no supervisadas, dado que Isolation Forest
   con dominancia empeoró (sección 10.3).
6. Redacción de resultados de tesis, documentando tanto los hallazgos positivos como la cadena de fugas de
   información detectadas y corregidas — en sí misma una contribución metodológica sobre reproducibilidad y
   validación honesta de resultados.
